# 52 — Composition showcase figures: blend vs vote vs schedule across n

Per-metric presentation figures for the composition methods at
n = 2, 3, 4 concurrent concepts and lambda in {0, 0.4}, on the balanced
30-group arms (n=2 balanced by notebook 51's top-up). GPU-free: merges
the existing logs (33, 39, 40, 51) and plots. Expression figures use the
raw mean joint delta — the improvement of the group's weakest concept —
see the companion doc plans/showcase-composition-explainer.md.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
METHODS = ("blend", "vote", "schedule")
COLORS = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}
SOURCES = [
    "resources/33_scaling3_bf16.jsonl",
    "resources/39_scaling4.jsonl",
    "resources/40_greedy_lambda.jsonl",
    "resources/51_scaling2_topup.jsonl",
]
SUMMARY_PATH = Path("resources/52_composition_summary.csv")
FIGURES = {
    (0.0, "joint"): "resources/52_lam0_expression.png",
    (0.0, "joint_cos"): "resources/52_lam0_token_cos.png",
    (0.4, "joint"): "resources/52_lam04_expression.png",
    (0.4, "joint_cos"): "resources/52_lam04_token_cos.png",
    (0.4, "ppl_ratio"): "resources/52_lam04_ppl.png",
}

In [ ]:
def rows_from(path, methods=METHODS):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c"), c.get("d")) if s]
        deltas = [r["expression"][s]["delta"] for s in synsets]
        cosines = [r["angles"][s]["delta"]["token_cos"] for s in synsets]
        rows.append(
            {
                "method": c["method"],
                "n": c["n"],
                "lam": c.get("lam", 0.0),
                "joint": min(deltas),
                "joint_cos": min(cosines),
                "all_positive": min(deltas) > 0,
                "all_positive_cos": min(cosines) > 0,
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


results = pd.concat([rows_from(p) for p in SOURCES], ignore_index=True)
cell_counts = results.groupby(["lam", "n"]).size()
print(cell_counts)
assert len(results) == 5400
assert (cell_counts == 900).all(), "expected 900 records per (lambda, n) cell"

In [ ]:
summary = (
    results.groupby(["method", "n", "lam"])
    .agg(
        joint=("joint", "mean"),
        joint_cos=("joint_cos", "mean"),
        all_positive=("all_positive", "mean"),
        all_positive_cos=("all_positive_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n_runs=("joint", "size"),
    )
    .reset_index()
)
summary.to_csv(SUMMARY_PATH, index=False)

# the n=3 and n=4 cells are untouched by the top-up - they must reproduce
# the published notebook-40 summary exactly
ref = pd.read_csv("resources/40_greedy_lambda_summary.csv")
check = summary[summary["n"] > 2].merge(
    ref[ref["n"] > 2], on=["method", "n", "lam"], suffixes=("", "_ref")
)
for col in ("joint", "joint_cos", "ppl_ratio"):
    assert (check[col] - check[f"{col}_ref"]).abs().max() < 1e-6, col
print(f"cross-check vs 40_greedy_lambda_summary: {len(check)} cells match")
summary.round(4)

In [ ]:
def line_figure(track, lam, ylabel, path, guardrail=None):
    sub = summary[summary["lam"] == lam]
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    for method in METHODS:
        s = sub[sub["method"] == method].sort_values("n")
        ax.plot(s["n"], s[track], marker="o", color=COLORS[method], label=method)
    if guardrail is None:
        ax.axhline(0, color="gray", lw=0.5)
    else:
        ax.axhline(1, color="gray", lw=0.5)
        ax.axhline(guardrail, color="gray", ls=":", lw=1)
    ax.set_xticks([2, 3, 4])
    ax.set_xlabel("concurrent concepts n")
    ax.set_ylabel(ylabel)
    ax.set_title(f"fluency weight $\\lambda$ = {lam}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


line_figure("joint", 0.0, "mean joint expression delta", FIGURES[(0.0, "joint")])
line_figure("joint_cos", 0.0, "mean joint token-cos gain", FIGURES[(0.0, "joint_cos")])

In [ ]:
line_figure("joint", 0.4, "mean joint expression delta", FIGURES[(0.4, "joint")])
line_figure("joint_cos", 0.4, "mean joint token-cos gain", FIGURES[(0.4, "joint_cos")])
line_figure(
    "ppl_ratio",
    0.4,
    "mean perplexity ratio",
    FIGURES[(0.4, "ppl_ratio")],
    guardrail=2.5,
)

In [ ]:
# combined fluency-cost figure: both lambda settings in one panel
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for lam, ls in ((0.0, "--"), (0.4, "-")):
    sub = summary[summary["lam"] == lam]
    for method in METHODS:
        s = sub[sub["method"] == method].sort_values("n")
        ax.plot(
            s["n"],
            s["ppl_ratio"],
            ls,
            marker="o",
            color=COLORS[method],
            label=f"{method}, $\\lambda$={lam}",
        )
ax.axhline(1, color="gray", lw=0.5)
ax.axhline(2.5, color="gray", ls=":", lw=1)
ax.set_xticks([2, 3, 4])
ax.set_xlabel("concurrent concepts n")
ax.set_ylabel("mean perplexity ratio")
ax.set_title("fluency cost: $\\lambda$ = 0 vs 0.4")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("resources/52_ppl_both_lambdas.png", dpi=150)
plt.show()

In [ ]:
# expression vs number of concepts, both fluency settings in one panel.
# lambda = 0 continuous, lambda = 0.4 dotted; color per method.
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for lam, ls in ((0.0, "-"), (0.4, ":")):
    sub = summary[summary["lam"] == lam]
    for method in METHODS:
        s = sub[sub["method"] == method].sort_values("n")
        ax.plot(
            s["n"],
            s["joint"],
            ls,
            marker="o",
            color=COLORS[method],
            label=f"{method}, $\\lambda$={lam}",
        )
ax.axhline(0, color="gray", lw=0.5)
ax.set_xticks([2, 3, 4])
ax.set_xlabel("concurrent concepts n")
ax.set_ylabel("mean joint expression delta")
ax.set_title("expression: $\\lambda$ = 0 vs 0.4")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("resources/52_expression_both_lambdas.png", dpi=150)
plt.show()

In [ ]:
for col in ("all_positive", "all_positive_cos"):
    pivot = summary.pivot_table(index="method", columns=["lam", "n"], values=col)
    print(f"\n{col} rate:")
    print(pivot.round(3).to_string())